# Sprawozdanie
## Environment Initialization

In [ ]:
environment = 'Local'
CHUNK_SIZE=200
CHUNK_OVERLAP=50
GOOGLE_DRIVE_PROJECT_DIRECTORY="./drive/MyDrive/Solvro/4"
ENVIRONMENTS = ['Local', 'Colab']
if environment not in ENVIRONMENTS:
    print(f"Invalid environment detected: \"{environment}\"!")
elif environment == 'Local':
    print(f"Path set up correctly for environment: \"{environment}\"!")
elif environment == 'Colab':
    from google.colab import drive
    from pathlib import Path
    !pip install -U langchain langchain-community langchain-text-splitters langchain-huggingface sentence-transformers chromadb langchain-chroma pymupdf opentelemetry-api opentelemetry-sdk
    !pip install langchain-google-genai
    drive.mount('/content/drive')
    base_path = Path(GOOGLE_DRIVE_PROJECT_DIRECTORY,)
    %cd {GOOGLE_DRIVE_PROJECT_DIRECTORY}
    !ls
    print(f"Path set up correctly for environment: \"{environment}\"!")

In [ ]:
from os import getenv
from dotenv import load_dotenv
from functools import wraps
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig

## API Key
### Przygotowanie API key
Do stworzenia agenta użyję model językowy przechowywany w chmurzę. Każde zapytanie do agenta będzie przesyłane przez internet do providera. Formą uwierzytelnienia mnie z serverem będzie personalny klucz nazywany **api key**'em.

Mój api key zapiszę w pliku `.env`. Dodaję `.env` do `.gitignore` by nikt inny się o nim nie dowiedział.

In [ ]:
# !echo "GEMINI_API_KEY=<mój-api-key-do-modeli-od-google-blablabla>"
# !echo ".env" >> .gitignore

### Ładowanie API key
Potrzebuje teraz wczytać klucz do programu.

In [ ]:
def load_api_key(environment_variable_name):
    _ = load_dotenv()
    api_key = getenv(environment_variable_name)
    if not api_key:
        raise ValueError(f"No key found: \"{environment_variable_name}\"!")
    return api_key

API_KEY = load_api_key("GEMINI_API_KEY")
print(f"Key loaded: \"GEMINI_API_KEY\"!")

## Dokumenty PDF
Z tytułu, iż jestem osamotnionym rozbitkiem na bezludnej wyspie, najbardziej interesuje mnie jedna następująca kwestia - jakie dania jestem w stanie przygotować z kokosów.
### Ładowanie PDFów do programu

In [ ]:
def load_pdf(pdf_path : str):
    loader = PyMuPDFLoader(file_path=pdf_path)
    return loader.load()

def load_pdfs(pdf_paths):
    pdfs_loaded = list()
    total_pages_loaded = 0
    for pdf_path in pdf_paths:
        pdf_loaded = load_pdf(pdf_path)
        total_pages_loaded += len(pdf_loaded)
        pdfs_loaded.extend(pdf_loaded)
    return total_pages_loaded, pdfs_loaded


pdf_paths = [
    './pdfy/babeczki_kokosowe.pdf', './pdfy/kokosanki.pdf',
    './pdfy/lody_kokosowe.pdf',
    './pdfy/kokosanka.pdf',
    './pdfy/likier_kokosowy.pdf'
]

total_pages_loaded, pdfs_loaded = load_pdfs(pdf_paths)
print(f"Total loaded pages: {total_pages_loaded}")

### Chunkowanie 
W celach czysto optymalizacyjnych, nie chcemy LLM-owi wysyłać wszystkich pdfów na raz, gdyż do utworzenia odpowiedzi model często będzie potrzebował kilku fragmentów. Po to jest nam chunking - dzielenie załadowanych pdfów na mniejsze fragmenty.

In [ ]:
def chunk_pdfs(loaded_pdfs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    docs = text_splitter.split_documents(pdfs_loaded)
    return docs
docs = chunk_pdfs(pdfs_loaded)
print("Loaded chunks:", len(docs))

### Utworzenie Bazy Wektorowej
Fajnie, że zbiór danych jest po-chunkowany. Ale żeby optymalizacja miała sens, musimy mniej więcej umieć przewidzieć, które fragmenty dokumentów będą potrzebne LLMowi do odpowiedzi (w przeciwnym razie biędzie trzeba wysłać wszystkie chunki!). Użyjemy tutaj mechanizmu generacji embeddingów. 

Embeddingi to numeryczna reprezentacja zawartości chunka. Zapytanie użytkownika także przetłumaczymy na wartość numeryczną i dzięki porównaniu wartości embeddingów jesteśmy w stanie numerycznie wyznaczych top X chunków które poruszają tematykę zawartą w pytaniu. W tym celu stworzymy bazę wektorową.

In [ ]:
def create_vector_store(docs):
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vectorstore = Chroma.from_documents(
        documents=docs,
        embedding=embeddings,
        # persist_directory="/content/drive/MyDrive/Colab Notebooks/ZadanieAgenty/"
        )
    return vectorstore

vector_store = create_vector_store(docs)

## Agent
### Narzędzia Agenta
Dzięki narzędzią model jest w stanie wykonać czynności z poza zakresu możliwości jakie ma na serwerze w chmurze. Oprócz kodu wykonawczego związanego z daną czynnością, funkcja musi posiadać komentarz opisujący swoje zastosowanie, a także kontekst, w którym narzędzie powinno być wywołane w sesji.

Do debugowania zrobie sobie dekorator:

In [ ]:
def log_function_usage(function):
    @wraps(function)

    def inner(*args, **kwargs):
        print(f"DEBUG: function called: \"{function.__name__}\".")
        return function(*args, **kwargs)

    return inner

#### `search_knowledge_base`
Podstawowe narzędzie, mające na celu udostępnienie zbioru naszych danych agentowi:

In [ ]:
@tool
@log_function_usage
def search_knowledge_base(query:str) -> str:
    """Searches the knowledge base for information from the loaded PDF documents.
    Use this tool when the user asks about the content of the documents.
    """
    retriever = vector_store.as_retriever(
        search_kwargs={"k": 10}
        )

    docs = retriever.invoke(query)

    if not docs:
        return "No matching information in the knowledge base."

    # Wyciągamy samą treść z każdego obiektu w docs i łączymy w jeden tekst
    return "\n\n---\n\n".join([doc.page_content for doc in docs])

#### `sing`

In [ ]:
@tool
@log_function_usage
def sing() ->str:
    """Sings facts about coconuts.
    Use if user mentions coconuts.
    """
    return """                      # piosenka Smokey Mountain - The coconut song (Da coconut song)
        Ko-ko-nut ko-ko-ko-ko-ko-nut ko-ko-nut (4x)
        Da kokonut nut is a giant nut
        If you eat too much, you'll get very fat
        Now, the kokonut nut is a big, big nut
        But this delicious nut is not a nut

        It's the coco fruit (it's the coco fruit)
        Of the coco tree (of the coco tree)
        From the coco palm fa-mi-ly

        There are so many uses of the kokonut tree
        You can build a bigger house for the family
        All you need is to find a kokonut man
        If he cuts the tree, he gets the fruit free

        It's the coco fruit (it's the coco fruit)
        Of the coco tree (of the coco tree)
        From the coco palm fa-mi-ly

        The kokonut bark for the kitchen floor
        If you save some of it, you can build a door
        Now, the kokonut trunk, do not throw this junk
        If you save some of it, you'll have a second floor

        The kokonut wood is very good
        It can stand 20 years if you pray it wood
        Now, the kokonut root, to tell you the truth
        You can throw it or use it as firewood

        The kokonut leaves good shade it gives
        For da roof, for da walls up against da teaves
        Now, the kokonut fruit, say my relatives
        Make good canon balls up against the thieves

        It's the coco fruit (it's the coco fruit)
        Of the coco tree (of the coco tree)
        From the coco palm fa-mi-ly

        Da kokonut nut is a giant nut
        If you eat too much, you'll get very fat,
        Now, the kokonut nut is a big, big nut
        But this delicious nut is not a nut (2x)

        It's the coco fruit (it's the coco fruit)
        Of the coco tree (of the coco tree)
        From the coco palm fa-mi-ly (3x)
    """

#### `translate_to_island_language`

In [ ]:
@tool
@log_function_usage
def translate_to_island_language(text: str) -> str:
    """Translates standard english into island language
    Use if user asks you to translate a specific english string into island language (język wyspiarski).
    """

    MORSE_CODE_DICT = {
        'A': '🥥🍌', 'B': '🍌🥥🥥🥥', 'C': '🍌🥥🍌🥥', 'D': '🍌🥥🥥', 'E': '🥥', 
        'F': '🥥🥥🍌🥥', 'G': '🍌🍌🥥', 'H': '🥥🥥🥥🥥', 'I': '🥥🥥', 'J': '🥥🍌🍌🍌', 
        'K': '🍌🥥🍌', 'L': '🥥🍌🥥🥥', 'M': '🍌🍌', 'N': '🍌🥥', 'O': '🍌🍌🍌', 
        'P': '🥥🍌🍌🥥', 'Q': '🍌🍌🥥🍌', 'R': '🥥🍌🥥', 'S': '🥥🥥🥥', 'T': '🍌', 
        'U': '🥥🥥🍌', 'V': '🥥🥥🥥🍌', 'W': '🥥🍌🍌', 'X': '🍌🥥🥥🍌', 'Y': '🍌🥥🍌🍌', 
        'Z': '🍌🍌🥥🥥', '1': '🥥🍌🍌🍌🍌', '2': '🥥🥥🍌🍌🍌', '3': '🥥🥥🥥🍌🍌', 
        '4': '🥥🥥🥥🥥🍌', '5': '🥥🥥🥥🥥🥥', '6': '🍌🥥🥥🥥🥥', '7': '🍌🍌🥥🥥🥥', 
        '8': '🍌🍌🍌🥥🥥', '9': '🍌🍌🍌🍌🥥', '0': '🍌🍌🍌🍌🍌', ',': '🍌🍌🥥🥥🍌🍌', 
        '.': '🥥🍌🥥🍌🥥🍌', '?': '🥥🥥🍌🍌🥥🥥', '/': '🍌🥥🥥🍌🥥', '-': '🍌🥥🥥🥥🥥🍌', 
        '(': '🍌🥥🍌🍌🥥', ')': '🍌🥥🍌🍌🥥🍌', '!': '🍌🥥🍌🥥🍌🍌', '\'': '🥥🍌🍌🍌🍌🥥'
    }
    
    encoded_words = []
    
    # split text to words
    for word in text.strip().split():
        encoded_chars = [
            MORSE_CODE_DICT[char] 
            for char in word.upper() 
            if char in MORSE_CODE_DICT
        ]
        if encoded_chars:
            encoded_words.append(" ".join(encoded_chars))
            
    # join letters with space, join words with /
    return " / ".join(encoded_words)

#### Poinformowanie agenta o narzędziach

In [ ]:
tools = [search_knowledge_base, sing, translate_to_island_language]

### Prompt Systemowy
Tutaj podaje kontekst w jakim agent ma się zachowywać.

In [ ]:
SYSTEM_PROMPT="""
Jesteś asystentem użytkownika, który właśnie stał się jednym z ocalałych pasażerów katastrofy Titanica, rozbitym na bezludnej wyspie. Masz za zadanie pomóc mu w czynnościach technicznych i odpowiadania na pytania dotyczące bazy danych jaka została ci udostępniona.
    1. Jeśli użytkownik zapyta cię o coś z bazy, użyj `search_knowledge_base`.
    2. Jeśli użytkownik wspomni o kokosach, użyj `sing` i zacytuj losowy fakt z piosenki.
By podnieść ducha świeżemu rozbitkowi, każdą wypowiedź zaczynaj słowami `Achoj!`, a wypowiedzi stylizuj mową piracką. Używaj emoji do wzbogacania odpowiedzi!
Jeśli zostaniesz poproszony o cytat, zacytuj dokładnie słowa z dokumentów. Nie rób Błędów!
"""

### Stworzenie Agenta
Tworzę agenta. Dodatkowo chcę zaimplementować pamięć konwersacji - agent powinien pamiętać wcześniejsze pytania i odpowiedzi. Używam tu funkcjonalności zapisu stanu poprzez checkpointer.

In [ ]:

class AgentSession:
    def __init__(self, config, model, system_prompt):
        self.config = config
        self.model = model
        self.system_prompt = system_prompt

        self.agent = create_agent(
            model=self.model,
            tools=tools,
            system_prompt=self.system_prompt,
            checkpointer=InMemorySaver()    # zapisuj stan, traktuj komunikację jako sesję
        )

    def query(self, query:str):
        result = self.agent.invoke({"messages": [{"role": "user", "content": query}]}, self.config)
        to_parse = result["messages"][-1]
        return f"\n+++++++\n{to_parse.content[0]['text']}\n-------\n"


In [ ]:
config: RunnableConfig = {"configurable": {"thread_id": "1"}} # zdefiniuj identyfikator sesji
agent = AgentSession(config, "google_genai:gemini-3.1-flash-lite", SYSTEM_PROMPT)

## Testy Zapytań
### Podstawowe pytanie o bazę

In [ ]:
print(agent.query("Przeszukaj bazę i kilkoma zdaniami opisz jej tematykę."))
print(agent.query("Jaki owoc gra główną rolę w zbiorze danych?"))

Agent poprawnie identyfikuje zbiór danych jako zbiór przepisów kulinarnych.

Agent poprawnie identifikuje owoc kokosu jako główny składnik pojawiający się w zbiorze danych.

(?) Agent czasami używa funkcji `sing` nie będąc poproszony? Potencjalnie niedopracowany docstring narzędzia `sing`?

### Sprawdzenie pamięci konwersacji

In [ ]:
print(agent.query("O co ciebie pytałem?"))

Agent poprawnie wyszukuje ostatnie zapytanie w konwersacji.

### Sprawdzenie działania narzędzia `sing`

In [ ]:
print(agent.query("Stary, ale bym się napił chłodnego kokosa!"))

Agent poprawnie używa narzędzia `sing`, używając go wedle specyfikacji kontekstu użycia.

### Sprawdzenie działania narzędzia `translate_to_island_language`

In [ ]:
print(agent.query("Przetłumacz następującą wiadomość na język wyspiarski: \"SOS! I need some help here!\""))

Agent poprawnie używa narzędzia `translate_to_island_language`, używając go wedle specyfikacji kontekstu użycia.
### Sprawdzenie wystąpienia halucynacji 

In [ ]:
print(agent.query("Powiedz my coś o meksykańskim chili z naszego zbioru danych!"))

Agent nie halucynuje informacji, które nie zawierają się w zbiorze danych.

### Sprawdzenie możliwości cytowania z bazy

In [ ]:
print(agent.query("Zacytuj mi dokument, gdzie jest wzmianka o babeczkach!"))

Agent jest w stanie zacytować fragment z zbioru danych.